# Ready-made pathological mesh cases

These files are **already generated**. This notebook is an add-on, not a replacement for `01_integrity_lab.ipynb`.

The controls are synthetic round-boss meshes, not GenCAD or SGS outputs. Canonical arrays are in **millimetres**; GLBs are in **metres**. All tests preserve indexed topology. The saved results are from an executed run. No inference or network call is made.

Open this notebook from its extracted folder using the existing CAD Integrity Lab environment. The first figures/readouts use saved diagnostic reports; the last two sections perform fresh audits and repairs.

In [1]:
import json
from pathlib import Path
import sys

# Locate the extracted fixture folder without changing the installed project.
choices = (Path.cwd(), Path.cwd() / "pathological-mesh-fixtures")
ROOT = next((p for p in choices if (p / "case_fixtures.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook with its extracted fixture folder as the working directory.")
sys.path.insert(0, str(ROOT))

from case_fixtures import load_arrays, load_brep, repair_policy, show_case
from IPython.display import display, Markdown

summary = json.loads((ROOT / "reports/summary.json").read_text())
rows = ["| Mesh | Triangles | Betti (F2) | Free edges | Winding conflicts | Nonmanifold vertices |",
        "|---|---:|---|---:|---:|---:|"]
for name, r in summary.items():
    rows.append(f"| `{name}` | {r['triangles']} | {tuple(r['betti_F2'])} | {r['boundary_edges']} | {r['orientation_conflicts']} | {len(r['nonmanifold_vertices'])} |")
display(Markdown("\n".join(rows)))

| Mesh | Triangles | Betti (F2) | Free edges | Winding conflicts | Nonmanifold vertices |
|---|---:|---|---:|---:|---:|
| `00_clean_boss` | 508 | (1, 0, 1) | 0 | 0 | 0 |
| `01_detached_reversed_cap` | 508 | (2, 0, 0) | 128 | 0 | 0 |
| `01_welded_not_oriented` | 508 | (1, 0, 1) | 0 | 64 | 0 |
| `01_repaired_cap` | 508 | (1, 0, 1) | 0 | 0 | 0 |
| `02_pinched_vertex` | 1016 | (1, 0, 2) | 0 | 0 | 1 |

## P1 — detached, reversed cap

The disk is present but separated by **0.0591 mm**. The cap is highlighted using the fault-injection record. The actual small gap is not visually exaggerated. Shared-edge winding conflicts only become detectable after the disconnected disk is reattached.

In [2]:
show_case("01_detached_reversed_cap").show(renderer="plotly_mimetype")

### Run the existing repair pipeline

This cell requires the `cad_integrity` package from the original project. It is an actual computation, not replay of the JSON report. Passing here means the declared **combinatorial** checks passed; it is not native-CAD certification.

In [3]:
from cad_integrity import RepairPipeline
import numpy as np

broken = load_brep("01_detached_reversed_cap")
result = RepairPipeline(repair_policy()).run(broken)
print("Decision:", result.report.decision)
print("Before (F2):", result.report.before.homology.betti_numbers)
print("After (F2):", result.report.after.homology.betti_numbers)
print("Free edges:", len(result.report.before.boundary_edge_ids), "->", len(result.report.after.boundary_edge_ids))
print("Maximum displacement (mm):", result.report.maximum_vertex_displacement)
print("Changes:", *result.report.changes, sep="\n  ")

assert result.report.decision == "topology_checks_passed"
v, f, _ = load_arrays("00_clean_boss")
assert np.array_equal(result.candidate.vertices, v)
canon = lambda triangle: tuple(np.roll(triangle, -int(np.argmin(triangle))))
assert [canon(result.candidate.face_vertices(i)) for i in range(result.candidate.face_count)] == [canon(t) for t in f]
print("Exact reference coordinates and oriented triangles restored: True")

Decision: topology_checks_passed
Before (F2): (2, 0, 0)
After (F2): (1, 0, 1)
Free edges: 128 -> 0
Maximum displacement (mm): 0.05906775770248984
Changes:
  Merged 64 vertices and 64 straight edges
  Reversed complete loops on 62 faces
Exact reference coordinates and oriented triangles restored: True


In [4]:
# Separately saved, verified output from this same pipeline and policy.
show_case("01_repaired_cap").show(renderer="plotly_mimetype")

## P2 — nonmanifold point contact

Two closed surfaces share one vertex (red marker). There are **no free edges**. The live pipeline must still refuse welding because the shared vertex is nonmanifold; there is no automatic repaired model for this case.

In [5]:
show_case("02_pinched_vertex").show(renderer="plotly_mimetype")

In [6]:
pinched = RepairPipeline(repair_policy()).run(load_brep("02_pinched_vertex"))
print("Decision:", pinched.report.decision)
print("Betti (F2):", pinched.report.before.homology.betti_numbers)
print("Free edges:", len(pinched.report.before.boundary_edge_ids))
print("Nonmanifold vertex IDs:", pinched.report.before.nonmanifold_vertex_ids)
print("Reason:", *pinched.report.errors, sep="\n  ")
assert pinched.report.decision == "rejected"
assert pinched.candidate is None
assert pinched.report.before.nonmanifold_vertex_ids == (4,)

Decision: rejected
Betti (F2): (1, 0, 2)
Free edges: 0
Nonmanifold vertex IDs: (4,)
Reason:
  Resolve invalid/duplicate/collapsed or nonmanifold input cells before welding


## Viewport files and intermediate stage

`meshes/` contains GLB, OBJ, and NPZ versions of all five stages. Use GLB for a 3D viewport, and the NPZ loader for exact indexed-topology analysis. `01_welded_not_oriented` is an additional **intermediate**, not a third pathological case: it has no free edges but still has 64 winding conflicts. There is intentionally no repaired P2 file.

Detailed observations and the 19-case validation record are in `reports/`. No geometry construction is required on your machine.